# From Literature to Benchmarks

**Where you are:** in pipeline 1 you found task clusters on the CPU and wrote a small catalogue of downstream tasks
by hand (outputs_pipeline_01/benchmark_catalogue.json). Those few papers are only the start.

**Goal of this pipeline:** search the **whole field pool** (all papers of your field) for
more papers that perform your tasks, decide for every paper whether it really does, back each decision with a
verbatim quote from the paper, and write an extended catalogue.

**The idea of this variant: your hand-reviewed papers are the search.** There are no keywords. Every paper in the
pool becomes an embedding vector, and the papers closest to your hand-reviewed papers (the **seeds**) are the
candidates. The LLM then decides which of them really perform the task.

This is a *guideline*, not a solution.

Given: setup, API client and helpers, reading the inputs, embedding the pool, fetching full
texts, and saving/checking the output.

**Rules of the API:** Every answer is cached, so rerunning a cell is free.

## 1. Setup
**Goal:** load the libraries and the API client, and set where files are read from and written to.

In [ ]:
# The API client (ILaaSClient), the text cleaner (strip_latex) and the quote checker (quote_match)
# come from the repository's helpers/ folder (see helpers/README.md).
import os, io, re, sys, json
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import zstandard   # reads the compressed corpus files

sys.path.insert(0, str(Path("../helpers").resolve()))   # this notebook lives in "suggested pipelines/"
import ilaas_client as h
from ilaas_client import ILaaSClient, parse_json_answer
from textutils import strip_latex, quote_match

pd.set_option("display.max_colwidth", 120)

In [ ]:
# Inputs: the INSPIRE release, the field pool and your hand-made catalogue from pipeline 1.
SRC = "your_path_to_the_INSPIRE_release"  # <- TODO: change this to your path
P1_DIR = "outputs_pipeline_01"
POOL_FILE = os.path.join(P1_DIR, "field_papers.jsonl")              # all papers of your field (pipeline 1)
REFERENCE_FILE = os.path.join(P1_DIR, "benchmark_catalogue.json")   # your hand-reviewed tasks (pipeline 1)

OUTPUT_DIR = "outputs_pipeline_02"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def out_path(name):
    """Full path of a file inside OUTPUT_DIR."""
    return os.path.join(OUTPUT_DIR, name)


# The API log and the answer cache go into OUTPUT_DIR.
h.LOG_DIR = Path(out_path("api_logs"))
h.LOG_FILE = h.LOG_DIR / "api_calls.jsonl"
h.CACHE_DIR = h.LOG_DIR / "cache"
h.LOG_DIR.mkdir(exist_ok=True)

In [ ]:
# models available on the ILaaS API

# Chat models:
#   gemma-4-31b                          : default
#   qwen-3.8-27b                        
#   qwen-3.6-35b-instruct                : unavailable since 17 Nov (listed, but every call fails)
#   llama-3.1-8b                         
#   qwen-3.6-35b-instruct-reasoning-high : unavailable since 17 Nov

# Reranker (/rerank):
#   bge-reranker-v2-m3  
               
# Embeddings (/embeddings):
#   bge-m3                               : 1024-dim vectors
MODEL = "gemma-4-31b"             # chat LLM (see API_NOTES.md: which model for what)
RERANK = "bge-reranker-v2-m3"     # scores how well a text fits a query (good at ranking)
EMBED = "bge-m3"                  # text -> vector (good at finding similar papers)
WORKERS = 4                       # at most 4 parallel requests per team
TEAM = "your_team"                # <-- your team name, written into the catalogue
FIELD = "heavy-ion"               # <-- your field, written into every catalogue row
PROMPT_VERSION = "v1"             # change it whenever you edit a prompt; write it into retrieval_method

In [ ]:
# API key: from the environment or from ~/.ilaas_api_key.
KEY_FILE = os.path.expanduser("~/.ilaas_api_key")
if not os.environ.get("ILAAS_API_KEY") and os.path.exists(KEY_FILE):
    os.environ["ILAAS_API_KEY"] = open(KEY_FILE).read().strip()

client = ILaaSClient(tag="pipeline_02_seed_only", use_cache=True, verbose=False)
if client.has_key:
    print("API reachable, models:", client.models())
else:
    print("No API key: only cached answers can be replayed.")

### API helpers (given)
- `chat_json(system, user)` — one LLM call that must answer with a JSON object; returns a dict (`{}` on failure).
- `pmap(fn, items)` — run `fn` over many items with at most `WORKERS` parallel calls.
- `rerank_scores(query, docs)` — reranker relevance (0–1) of every doc for one query.
- `embed_texts(texts)` — normalised `bge-m3` vectors.

In [ ]:
FAILED_CALLS = []   # calls that failed after all retries; decide what happens to their papers (e.g. -> uncertain)


def as_text(v):
    """Models sometimes return a list or a number where a string was asked for."""
    if v is None:
        return ""
    if isinstance(v, list):
        return " ".join(as_text(x) for x in v)
    return v if isinstance(v, str) else str(v)


def chat_json(system, user, max_tokens=1500, keep_lists=()):
    """One JSON answer. A failed call returns {} instead of stopping the whole batch.
    Values are turned into strings, except the keys in keep_lists."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]
    try:
        r = client.chat(MODEL, msgs, response_format={"type": "json_object"}, temperature=0,
                        max_tokens=max_tokens, thinking=False)
    except Exception as e:
        FAILED_CALLS.append(f"{type(e).__name__}: {str(e)[:200]}")
        return {}
    ans = parse_json_answer(r["choices"][0]["message"].get("content") or "") or {}
    return {k: (v if k in keep_lists and isinstance(v, list) else as_text(v)) for k, v in ans.items()}


def pmap(fn, items):
    """fn over items with <= WORKERS parallel API calls; results in input order."""
    with ThreadPoolExecutor(WORKERS) as ex:
        return list(ex.map(fn, items))


def rerank_scores(query, docs, batch=200):
    """Reranker relevance of every doc for the query (same order as docs)."""
    parts = [docs[i:i + batch] for i in range(0, len(docs), batch)]
    res = pmap(lambda part: client.rerank(RERANK, query, part, top_n=len(part)), parts)
    out = []
    for part, r in zip(parts, res):
        s = [0.0] * len(part)
        for x in r["results"]:
            s[x["index"]] = x["relevance_score"]
        out += s
    return out


def embed_texts(texts, batch=64):
    """bge-m3 vectors (normalised to length 1) for a list of texts."""
    batches = [texts[i:i + batch] for i in range(0, len(texts), batch)]
    res = pmap(lambda b: client.embed(EMBED, b), batches)
    v = np.array([d["embedding"] for r in res for d in sorted(r["data"], key=lambda d: d["index"])], dtype=np.float32)
    return v / np.linalg.norm(v, axis=1, keepdims=True)

In [ ]:
# Try the API once (this answer is cached).
print(chat_json("Return only JSON.", 'What is the capital of France? Answer as {"answer": "..."}'))
print(rerank_scores("jet quenching", ["Jet quenching in Pb-Pb collisions", "Neutrino oscillations at T2K"]))

## 2. Load the field pool
**Goal:** load all papers of your field and clean their text. You search in this pool.

In [ ]:
pool_df = pd.read_json(POOL_FILE, lines=True, dtype={"recid": str})
pool_df = pool_df[pool_df["abstract"].notna() & (pool_df["abstract"].str.strip() != "")]
pool_df = pool_df.drop_duplicates(subset="recid").reset_index(drop=True)

# strip_latex removes LaTeX, MathML and HTML (tested on the corpus).
pool_df["title"] = pool_df["title"].fillna("").map(strip_latex)
pool_df["abstract"] = pool_df["abstract"].map(strip_latex)
pool_df["year"] = pool_df["year"].astype(str).str[:4]

by_id = pool_df.set_index("recid", drop=False)   # look up a paper by recid: by_id.loc["1234567"]
print(f"pool: {len(pool_df)} papers with abstract")

## 3. Define the tasks
**Goal:** write every task so that an LLM can check a paper against it: INPUT → TARGET in a scientific setting.

The tasks and your hand-reviewed papers are read from pipeline 1. Then it is up to you how to describe
each task to the LLM. Things that matter a lot:
- name the input and the target explicitly, not just the task name;
- say what does **not** count (look-alikes: same words, different task — e.g. a jet-suppression measurement is not
  "classify individual jets as quenched");
- a short query sentence for the reranker can be different from the description for the LLM.

In [ ]:
reference = json.load(open(REFERENCE_FILE))   # {"team": ..., "tasks": [{"task_id": ..., "papers": [rows]}]} from pipeline 1
ref_df = pd.DataFrame([row for t in reference["tasks"] for row in t["papers"]])
# recid as a string (the pool's ids are strings); from paper_id if a row has no recid
rid = ref_df["recid"] if "recid" in ref_df else pd.Series("", index=ref_df.index)
ref_df["recid"] = rid.fillna("").astype(str).str.replace(r"\.0$", "", regex=True)
ref_df.loc[ref_df["recid"] == "", "recid"] = ref_df["paper_id"].str.replace("INSPIRE:", "", regex=False)
print("hand-reviewed papers per task:", ref_df["task_id"].value_counts().to_dict())

# Everything you wrote per task, as a starting point.
for task_id, g in ref_df.groupby("task_id"):
    print(f"\n== {task_id}: {g['task_label'].iloc[0]} ({g['task_type'].iloc[0]})")
    print("   inputs: ", sorted(set(g["input_modality"])))
    print("   targets:", sorted(set(g["observable_or_target"])))

In [ ]:
# TODO: build one entry per task. Keep at least these keys (they are written into the catalogue rows);
#       add whatever your prompts and search need (description, look-alikes, short query, ...).
TASKS = {
    # "example_task_id": {
    #     "task_label": "...",
    #     "task_type": "classification",
    #     "fm_scope": "...",
    #     "input_modality": "...",
    #     "observable_or_target": "...",
    #     "setting": "...",            (optional: system, energy, experiment)
    #     "description": "... NOT this task: ...",
    #     "short_query": "one sentence for the reranker",
    # },
}

## 4. Retrieval: seed and expand
**Goal:** 50–200 candidate papers per task, so that (almost) all real papers on the task are among them.
Wrong candidates are removed later.

How seed and expand works:
1. The hand-reviewed papers of a task are the **seeds**. Every pool paper gets a similarity to the seeds
   (cosine = dot product, because the vectors have length 1).
2. The seeds + the `N_NEIGHBOURS` most similar papers are the candidates.
3. The reranker scores the candidates against the task's `short_query`. Embeddings are good at *finding* similar
   papers but poor at *ordering* them; the reranker is good at ordering.

Things to decide:
- **Similarity to which seed?** To the *closest* seed (max over seeds) keeps papers that resemble only one of your
  seeds. The *mean* over seeds favours papers in the middle of the task and loses the unusual ones.
- **How many neighbours?** Look at the similarity of the last neighbour you keep. If it is still high, real papers
  are probably further down: take more.
- **A task without seeds in the pool:** embed the `short_query` sentence and use it as the seed.
- **What to embed:** embedding the whole pool takes a while. You may embed only the relevant part of it
  (`SCOPE_RX` below, e.g. papers that mention jets). Your seeds are always embedded.
- **Write how a paper was found into `retrieval_method`**, e.g. `"seed-and-expand"`,
  and `"seed"` for the hand-reviewed papers themselves.

In [ ]:
# Embed the pool once (given). The vectors are cached in EMB_FILE; later runs only embed papers that are new.
SCOPE_RX = None   # optional: only embed papers whose title or abstract matches, e.g. re.compile(r"\bjets?\b")
EMB_FILE = out_path("emb_bge-m3_pool.npz")

# text of every paper: for the embedding model and for the reranker
docs = {r: f"{t}. {a}" for r, t, a in zip(pool_df["recid"], pool_df["title"], pool_df["abstract"])}

if SCOPE_RX is None:
    scope_ids = set(pool_df["recid"])
else:
    hit = pool_df["title"].str.lower().str.contains(SCOPE_RX) | pool_df["abstract"].str.lower().str.contains(SCOPE_RX)
    scope_ids = set(pool_df["recid"][hit])
scope_ids = sorted(scope_ids | (set(ref_df["recid"]) & set(by_id.index)))   # your seeds are always embedded

emb = {}
if os.path.exists(EMB_FILE):
    z = np.load(EMB_FILE)
    emb = dict(zip(map(str, z["ids"]), z["vecs"]))
todo_ids = [r for r in scope_ids if r not in emb]
if todo_ids:
    print(f"embedding {len(todo_ids)} papers ...")
    emb.update(zip(todo_ids, embed_texts([docs[r][:8000] for r in todo_ids])))   # <= 8192 tokens per text
    np.savez(EMB_FILE, ids=np.array(list(emb)), vecs=np.array(list(emb.values()), dtype=np.float32))

E_ids = scope_ids                                       # row i of E is the paper E_ids[i]
E = np.array([emb[r] for r in E_ids], dtype=np.float32)  # vectors have length 1: E @ v = cosine similarity
pos = {r: i for i, r in enumerate(E_ids)}               # recid -> row in E
print(f"{len(E_ids)} papers embedded | seeds not in the pool:",
      sorted(set(ref_df["recid"]) - set(by_id.index)) or "none")

In [ ]:
N_NEIGHBOURS = 100   # TODO: how many similar papers per task go on to screening?


def find_candidates(task_id, task):
    """DataFrame of candidates for one task, best first, with at least the columns
    recid, title, retrieval_method (how the paper was found — this goes into the catalogue).
    Useful extra columns: similarity, closest_seed, rerank_score."""
    seeds = [r for r in ref_df.loc[ref_df["task_id"] == task_id, "recid"] if r in pos]
    # TODO 1: the seed vectors E[[pos[r] for r in seeds]]; no seed -> embed_texts([task["short_query"]])
    # TODO 2: similarity of every paper to the seeds (E @ S.T), and the N_NEIGHBOURS best that are not seeds
    # TODO 3: reranker scores for seeds + neighbours: rerank_scores(task["short_query"], [docs[r] for r in ...])
    # TODO 4: one row per paper, with retrieval_method "seed" or "seed-and-expand (...)"
    raise NotImplementedError


cands = {}   # TODO: cands[task_id] = find_candidates(task_id, task) for every task, and save them as CSV

In [ ]:
# TODO: look at the candidates of one task: does the similarity drop sharply somewhere? are the last ones still
#       on topic? (if yes: raise N_NEIGHBOURS)
...

## 5. Screening (title + abstract) → review queue
**Goal:** sort the candidates into `candidate` / `uncertain` / `excluded` from the abstract, and keep the best 20–30
per task for the (more expensive) verification.

Write your own prompts. Things we learned:
- ask for a **JSON** answer with a fixed set of labels and a one-sentence reason;
- "mentions only" is the common trap: the task is used as an ingredient, motivation or future plan;
- LLMs almost never say "uncertain" by themselves. Asking twice with two *different* prompts and calling a split
  vote "uncertain" works better than asking once. Two prompts that worked:
  - **A:** four labels — performs the task / mentions it only / different task (or the same words for another task) /
    uncertain;
  - **B:** first write down the paper's *input* and *target*, then decide (performs / does not perform / uncertain);
- optional queue order: papers both prompts accept first, then the split votes, then by reranker score;
- set `PROMPT_VERSION` and write it into `retrieval_method`, so you know which prompt produced a row.

In [ ]:
SCREEN_SYSTEM = """TODO: your instructions for the LLM"""
SCREEN_A = """TODO: the task ({task_id}) and the paper ({title} / {abstract}), and the JSON format to answer in"""
SCREEN_B = """TODO: a second, different prompt (e.g. extract the paper's input and target first, then decide)"""


def screen_one(job):
    """job = (task_id, recid) -> the answers of both prompts, as dicts."""
    task_id, rid = job
    # TODO: fill both prompts and call chat_json(...) twice
    raise NotImplementedError


# TODO: screen all candidates (use pmap), map the answers to candidate / uncertain / excluded,
#       call a split vote "uncertain", and build queues[task_id] = DataFrame of the 20-30 papers per task
#       that go on to verification.
queues = {}

In [ ]:
# TODO: how did the screening judge your seeds? They are in the candidate list, so this is a free test:
#       they should come out as candidate or uncertain. A seed screened as excluded = your prompt is too strict.
...

## 6. Fetch the full texts of the queue papers (given)
**Goal:** get the full text of the queue papers only. The full-text files are large (~5 GB compressed per decade),
so they are scanned once and only the queue papers are saved to `fulltext_queue.jsonl`; later runs only fetch papers
that are new in the queue. 

Result: `fulltext[recid] = {"recid", "arxiv_id", "sections": [{"heading", "text"}, ...]}`.

In [ ]:
USE_FULLTEXT = True
FT_FILE = out_path("fulltext_queue.jsonl")
RECID_RX = re.compile(r'^\{"recid": ?"?(\d+)')

fulltext = {}
if os.path.exists(FT_FILE):
    fulltext = {r["recid"]: r for r in map(json.loads, open(FT_FILE))}
need = {r for q in queues.values() for r in q["recid"]} - set(fulltext)
DECADES = ["pre1990", "1990s", "2000s", "2010s", "2020s"]


def decade_of(year):
    """Corpus folder of a paper: the corpus is split by the year of earliest_date; all years before 1990 are in pre1990."""
    y = str(year)[:4]
    if not y.isdigit():
        return None
    return "pre1990" if int(y) < 1990 else y[:3] + "0s"


def read_zst_lines(path):
    """Yield the lines of a compressed .jsonl.zst file, unpacked on the fly (nothing is written to disk)."""
    with open(path, "rb") as fh:
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(fh)
        yield from io.TextIOWrapper(reader, encoding="utf-8")


if USE_FULLTEXT and need:
    # the decades of the queue papers; a paper without a usable year makes every decade a candidate
    dec_of = {r: decade_of(by_id.at[r, "year"]) for r in need}
    decades = [d for d in DECADES if d in dec_of.values() or None in dec_of.values()]
    with open(FT_FILE, "a", encoding="utf-8") as out:
        for dec in decades:
            if not need:
                break
            path = os.path.join(SRC, dec, f"fulltext_{dec}.jsonl.zst")
            if not os.path.exists(path):
                print("! missing", path, "- the queue papers of this decade get no full text (download it, or fix SRC)")
                continue
            print("scanning", path, "for", len(need), "papers ...")
            for line in read_zst_lines(path):
                m = RECID_RX.match(line)
                if not m or m.group(1) not in need:
                    continue
                r = json.loads(line)
                sections = [{"heading": strip_latex(s.get("heading") or ""), "text": strip_latex(s.get("text") or "")}
                            for s in r.get("sections") or []]
                rec = {"recid": m.group(1), "arxiv_id": r.get("arxiv_id") if r.get("arxiv_id") not in (None, "None", "") else "",
                       "sections": [s for s in sections if s["text"]]}
                out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                fulltext[rec["recid"]] = rec
                need.discard(rec["recid"])
                if not need:
                    break
queue_ids = {r for q in queues.values() for r in q["recid"]}
print(f"full text available for {len(queue_ids & set(fulltext))} of {len(queue_ids)} queue papers")

## 7. Verification with evidence
**Goal:** decide every queue paper again, now with the full text, and get a **verbatim quote** as evidence.

Things to decide:
- **What text does the LLM see?** Whole papers are long and expensive. Cutting the full text into chunks and
  letting the reranker pick the few most relevant ones is much cheaper and works about as well (in the tests:
  ~1 500-character chunks, the best 3 + the abstract).
- **What does it return?** At least: a decision, `match_rationale`, `study_type`, and `evidence_text` copied word for
  word. Optionally input/target in the paper's own words, metric, reported result, dataset, code.
- **Don't trust the LLM — check it:** `quote_match(quote, text)` tells you whether a quote is really in the paper
  (`"exact"`, `"normalized"`, `"fragments"`, `"not_found"`, `"empty"`). Search the quote in the abstract and in every
  section to get the real `evidence_location`, instead of believing the location the LLM names.
- **Final decision rule:** combine verification, screening and the quote check into
  `supported` / `candidate` / `uncertain` / `excluded`. E.g. what if screening and verification disagree?
  What if the LLM says "performs the task" but the quote is not in the paper?

In [ ]:
print(quote_match("jets are quenched", "In Pb-Pb collisions jets are quenched by the medium."))
print(quote_match("jets are strongly quenched", "In Pb-Pb collisions jets are quenched by the medium."))

In [ ]:
VERIFY_SYSTEM = """TODO"""
VERIFY_USER = """TODO"""


def verify_one(job):
    """job = (task_id, recid) -> dict with the LLM answer plus your checks (quote found? where?)."""
    task_id, rid = job
    # TODO
    raise NotImplementedError


# TODO: verify all queue papers (pmap) and build `cat`: a DataFrame with one row per (task, paper) and at least
#       the FIELDS of section 9. Keep extra diagnostic columns if useful (recid, screening answer, quote_match, ...).
cat = ...

## 8. Clean-up: duplicates, your hand-reviewed rows, human review
**Goal:** a catalogue without double counting.
- **Duplicates:** the same study often appears twice (paper + proceedings, thesis + paper, two INSPIRE records).
  Give all versions the same `study_group_id` and keep only one of them per task. You already have the
  embeddings from section 4 (`emb`), so this costs nothing: e.g. cosine ≥ 0.95 = same study, 0.90–0.95 = list
  the pair for a human to check.
- **Your rows from pipeline 1** are `human_reviewed`: they should win over the automated row for the same paper,
  and be added if the pipeline missed them.
- **Human review:** every automated row stays `automated_proposal` until someone has read it. A practical way:
  export the rows to a CSV, write your decision into a column, read the CSV back and apply it.
  Start with the `uncertain` rows.

In [ ]:
# TODO: study_group_id + remove duplicate versions (reuse emb from section 4)
...

In [ ]:
# TODO: merge your hand-reviewed rows from ref_df
...

In [ ]:
# TODO: export a review CSV, read your decisions back, set review_status = "human_reviewed"
...

## 9. Export and success check (given)
**Goal:** write the catalogue as JSON with exactly these fields per paper, and check the minimum success criterion.

| field | description | note |
|---|---|---|
| `field` | Application area / Scientific field (e.g. pp, PbPb, GW, multimessenger) | Do not collapse pp/PbPb |
| `fm_scope` | Intended foundation-model input/context | E.g. particle-level jet constituents |
| `task_id` | Stable short identifier for the task | E.g. jet_classification_pp |
| `task_label` | Human-readable task name | |
| `task_type` | classification / regression / reconstruction / detection / parameter estimation / generation / other | |
| `observable_or_target` | What is predicted or measured | Use unknown only when source is genuinely insufficient |
| `input_modality` | What goes in | |
| `study_type` | experimental / phenomenological / methodological / unspecified | Where relevant; otherwise unspecified |
| `paper_id` | Paper identifier (arXiv ID, DOI, or INSPIRE key) | |
| `paper_url` | Direct link to paper | |
| `study_group_id` | Groups duplicate versions of the same study | |
| `match_rationale` | Why this study supports a potential benchmark for this FM scope | Required for every row |
| `evidence_text` | Quoted passage supporting the match | Required for supported decisions |
| `evidence_location` | Where in the paper (abstract / section / page) | |
| `decision` | candidate / supported / uncertain / excluded | |
| `review_status` | automated_proposal or human_reviewed | |
| `retrieval_method` | Retrieval method (e.g. lexical, LLM-assisted) and the query family that produced this row | For reproducibility |

Optional: `recid` (INSPIRE record id, makes the quote check exact), `setting` (system, energy, experiment), `title`,
`paper_aliases`, and the enrichment fields `dataset_status`, `dataset_url`, `model_code_status`, `model_code_url`,
`metric`, `reported_result`, `evaluation_setting`. The file is written as `{"team": ..., "tasks": [{"task_id": ...,
"papers": [rows]}]}`; check it with `python3 ../catalogue/validate.py <file>` (add `--corpus` to check the quotes).

In [ ]:
FIELDS = ["field", "fm_scope", "task_id", "task_label", "task_type", "observable_or_target", "input_modality",
          "study_type", "paper_id", "paper_url", "study_group_id", "match_rationale", "evidence_text",
          "evidence_location", "decision", "review_status", "retrieval_method"]

ALLOWED = {
    "task_type": {"classification", "regression", "reconstruction", "detection", "parameter estimation",
                  "generation", "other"},
    "study_type": {"experimental", "phenomenological", "methodological", "unspecified"},
    "decision": {"candidate", "supported", "uncertain", "excluded"},
    "review_status": {"automated_proposal", "human_reviewed"},
}
# optional fields (validate.py and the scorer know them; leave them out if you don't have them)
OPTIONAL = ["recid", "setting", "title", "paper_aliases", "dataset_status", "dataset_url", "dataset_name",
            "model_code_status", "model_code_url", "model_code_name", "metric", "reported_result",
            "evaluation_setting", "review_comment"]


def check_row(row):
    """Return a list of problems in one row (empty list = OK)."""
    problems = [f"missing field {f}" for f in FIELDS if f not in row]
    problems += [f"{f}={row.get(f)!r} not in {sorted(v)}" for f, v in ALLOWED.items() if row.get(f) not in v]
    if not row.get("match_rationale"):
        problems.append("match_rationale is required")
    if row.get("decision") == "supported" and not row.get("evidence_text"):
        problems.append("evidence_text is required for supported rows")
    return problems


def success_check(cat):
    """The minimum success criterion of the hackathon."""
    ok = cat[(cat["decision"] == "supported") & (cat["review_status"] == "human_reviewed")]
    live = cat[cat["decision"] != "excluded"]
    checks = {
        "at least 2 distinct task_id": cat["task_id"].nunique() >= 2,
        "each task: >= 1 supported AND human_reviewed row": set(ok["task_id"]) >= set(cat["task_id"]),
        "every row has paper_id and paper_url": bool((cat["paper_id"] != "").all() and (cat["paper_url"] != "").all()),
        "no duplicate study within a task (non-excluded rows)": not live.duplicated(["task_id", "study_group_id"]).any(),
        "supported rows have evidence_text": bool((cat.loc[cat["decision"] == "supported", "evidence_text"] != "").all()),
    }
    for name, passed in checks.items():
        print(f"  [{'x' if passed else ' '}] {name}")
    return checks


def save_catalogue(cat, name="catalogue.json"):
    """Check every row and write {"team": ..., "tasks": [{"task_id": ..., "papers": [rows]}]} (the layout
    validate.py and the scorer read), with FIELDS and the optional fields you filled per paper."""
    cat = cat.astype(object).where(cat.notna(), "")
    for _, r in cat.iterrows():
        for p in check_row(r.to_dict()):
            print(f"  [{r.get('task_id')} | {r.get('paper_id')}] {p}")
    keep = FIELDS + [f for f in OPTIONAL if f in cat.columns]
    tasks = []
    for task_id, g in cat.groupby("task_id"):
        papers = [{k: v for k, v in r.items() if k in FIELDS or v != ""} for r in g[keep].to_dict(orient="records")]
        tasks.append({"task_id": task_id, "task_label": g["task_label"].iloc[0],
                      "n_papers": int((g["decision"] != "excluded").sum()),
                      "decisions": g["decision"].value_counts().to_dict(), "papers": papers})
    with open(out_path(name), "w", encoding="utf-8") as f:
        json.dump({"team": TEAM, "tasks": tasks}, f, indent=1, ensure_ascii=False, default=str)
    print("saved", len(cat), "rows in", len(tasks), "tasks to", os.path.abspath(out_path(name)))

In [ ]:
# Run when `cat` is ready:
# success_check(cat)
# save_catalogue(cat)

## 10. Error report
**Goal:** write down, in a few sentences, where your pipeline goes wrong. Count for example: how many candidates
were a different task with the same words, how many screening votes were split, how many quotes were not found,
how many papers had no full text, how many API calls failed.
Seed-specific: how many seeds were screened as excluded, how similar the last neighbour still was, and how many
papers you know of (from reading, citations, colleagues) the neighbours missed. These numbers tell you what to improve next.

In [ ]:
# TODO
...